In [1]:
# === Cell 1: imports ===
from pathlib import Path
import re, gc, joblib, fiona, rasterio, time
from datetime import datetime
import numpy as np
import pandas as pd
from rasterio.transform import rowcol
from rasterio.windows import Window
from joblib import Parallel, delayed
from scipy.ndimage import uniform_filter, binary_dilation
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from xgboost import XGBRFRegressor
from rasterio.warp import transform as crs_transform

In [2]:
# === Cell 2: make bathy raster, grid raster, filter raster, and csv for model ===
def list_tifs(d:Path): return sorted(p for p in d.glob("*.tif") if p.is_file())
def s2_code(p:Path):
    s=p.stem.split("_")
    if len(s)<3 or s[0].upper()!="S2": raise ValueError(f"Bad S2 name (expected S2_<CODE>_...): {p.name}")
    return s[1].upper()
def bathy_index(d:Path):
    idx={}
    for shp in d.glob("*_bathy_prj.shp"): idx[shp.stem.split("_")[0].upper()]=shp
    return idx
# Mean depth per S2 pixel (float32, nodata=-1) on the ORIGINAL S2 grid/CRS (STOP if CRS mismatch)
def bathy_mean_raster(shp:Path,s2:Path,out_dir:Path,site:str,nodata=-1,n_jobs=-1,
                      chunk=100000,min_depth=0,max_depth=20)->Path:
    out=out_dir/f"Bathy_rast_{site}.tif"
    mn=min_depth if isinstance(min_depth,(int,np.integer,float,np.floating)) else None
    mx=max_depth if isinstance(max_depth,(int,np.integer,float,np.floating)) else None
    with rasterio.open(s2) as r:
        tr,crs,w,h=r.transform,r.crs,r.width,r.height
        if crs is None: raise ValueError(f"S2 raster has no CRS: {s2.name}")
        s2w=crs.to_wkt() if hasattr(crs,"to_wkt") else str(crs)
    flats=[]; vals=[]
    with fiona.open(shp) as src:
        dk=next((k for k in src.schema["properties"] if k.lower()=="depth"),None)
        if dk is None: raise ValueError(f"No Depth/depth column found in: {shp.name}")
        shpw=src.crs_wkt or src.crs
        if shpw is None: raise ValueError(f"Bathy shapefile has no CRS: {shp.name}")
        shpw=str(shpw)
        if s2w!=shpw:
            print(f"CRS MISMATCH -> STOP\n  S2:   {s2.name}\n  CRS:  {s2w}\n  Bathy:{shp.name}\n  CRS:  {shpw}")
            raise SystemExit("Stop: bathy CRS != S2 CRS (no reprojection requested).")
        for f in src:
            g=f.get("geometry"); 
            if not g: continue
            v=f["properties"].get(dk)
            if v is None: continue
            try: v=abs(float(v))
            except: continue
            if mn is not None and v<mn: continue
            if mx is not None and v>mx: continue
            t=g.get("type"); c=g.get("coordinates"); pts=[c] if t=="Point" else (c if t=="MultiPoint" else [])
            for x,y in pts:
                rr,cc=rowcol(tr,x,y)
                if 0<=rr<h and 0<=cc<w: flats.append(rr*w+cc); vals.append(v)
    n=len(flats); sumflat=np.zeros(h*w,np.float32); cntflat=np.zeros(h*w,np.uint32)
    if n:
        flats=np.asarray(flats,np.int64); vals=np.asarray(vals,np.float32)
        def red(fa,va):
            u,inv=np.unique(fa,return_inverse=True)
            return u,np.bincount(inv,weights=va).astype(np.float32),np.bincount(inv).astype(np.uint32)
        if n_jobs!=1 and n>chunk:
            for u,su,cu in Parallel(n_jobs=n_jobs)(delayed(red)(flats[i:min(i+chunk,n)],vals[i:min(i+chunk,n)]) for i in range(0,n,chunk)):
                np.add.at(sumflat,u,su); np.add.at(cntflat,u,cu)
        else:
            u,su,cu=red(flats,vals); np.add.at(sumflat,u,su); np.add.at(cntflat,u,cu)
    arr=np.full((h,w),nodata,np.float32); cnt=cntflat.reshape(h,w); m=cnt>0
    if m.any(): arr[m]=(sumflat.reshape(h,w)[m]/cnt[m]).astype(np.float32)
    prof=dict(driver="GTiff",height=h,width=w,count=1,dtype="float32",crs=crs,transform=tr,nodata=nodata,compress="ZSTD",predictor=2,BIGTIFF="IF_SAFER")
    with rasterio.open(out,"w",**prof) as dst: dst.write(arr,1)
    return out
# 1 km grid snapped to bathy raster (random ints 1..5), clipped to same extent
def rand_1km_grid(bathy_rast:Path,out_dir:Path,site:str,cell_m=1000,seed=100)->Path:
    out=out_dir/f"1kmGrid_{site}.tif"
    with rasterio.open(bathy_rast) as src: tr,crs,w,h=src.transform,src.crs,src.width,src.height
    rx,ry=abs(tr.a),abs(tr.e); bx=max(1,int(round(cell_m/rx))); by=max(1,int(round(cell_m/ry)))
    gw,gh=(w+bx-1)//bx,(h+by-1)//by
    rng=np.random.default_rng(seed); small=rng.integers(1,6,size=(gh,gw),dtype=np.int16).astype(np.int8)
    arr=np.repeat(np.repeat(small,by,0),bx,1)[:h,:w].astype(np.int8)
    prof=dict(driver="GTiff",height=h,width=w,count=1,dtype="int8",crs=crs,transform=tr,nodata=-1,compress="ZSTD",predictor=2,BIGTIFF="IF_SAFER")
    with rasterio.open(out,"w",**prof) as dst: dst.write(arr,1)
    return out
# Window-mean sampling (mirror padding) + exclusion buffer around any S2 invalid pixel
def sample_rasts_with_avgs(s2:Path,bathy_rast:Path,grid_1km:Path,kernel_sizes=(3,5,7,9,11,13,15),max_bands=5,bathy_nodata=-1,min_depth=0,max_depth=20)->pd.DataFrame:
    ks=list(kernel_sizes)
    mn=min_depth if isinstance(min_depth,(int,np.integer,float,np.floating)) else None
    mx=max_depth if isinstance(max_depth,(int,np.integer,float,np.floating)) else None
    with rasterio.open(bathy_rast) as bsrc:
        depth=bsrc.read(1).astype(np.float32)
        bnod=bsrc.nodata if bsrc.nodata is not None else bathy_nodata
        depth_ok=(depth!=bnod)
        if mn is not None: depth_ok&=(depth>=mn)
        if mx is not None: depth_ok&=(depth<=mx)
        h,w=depth.shape
    with rasterio.open(s2) as ssrc:
        if (ssrc.height,ssrc.width)!=(h,w):
            raise ValueError(f"Shape mismatch bathy={h,w} vs s2={ssrc.height,ssrc.width}")
        nb=min(ssrc.count,int(max_bands)) if max_bands else ssrc.count
        nod=ssrc.nodata
        # keep sample if center pixel is valid in every used band
        center_ok=np.ones((h,w),bool)
        for bi in range(1,nb+1):
            b=ssrc.read(bi).astype(np.float32)
            m=(ssrc.read_masks(bi)==0)|(~np.isfinite(b))
            if nod is not None and not (isinstance(nod,float) and np.isnan(nod)):
                m|=(b==nod)
            center_ok&=~m
            del b
        ok=depth_ok & center_ok
        idx=np.flatnonzero(ok.ravel())
        cols=["group","depth","lon","lat"]+[f"B{i}" for i in range(1,nb+1)]+[f"B{i}_W{k}" for i in range(1,nb+1) for k in ks]
        print(f"Samples kept: {idx.size} (depth_ok={int(depth_ok.sum())}, center_bad={int((~center_ok).sum())}, kept={int(ok.sum())})")
        if idx.size==0:
            return pd.DataFrame(columns=cols)
        with rasterio.open(grid_1km) as gsrc:
            out={"group":gsrc.read(1).ravel()[idx].astype(np.int16),"depth":depth.ravel()[idx].astype(np.float32)}
        r,c=np.unravel_index(idx,(h,w))
        x,y=rasterio.transform.xy(ssrc.transform,r,c,offset="center")
        out["lon"],out["lat"]=map(np.asarray,crs_transform(ssrc.crs,"EPSG:4326",x,y))
        for bi in range(1,nb+1):
            b=ssrc.read(bi).astype(np.float32)
            vm=(ssrc.read_masks(bi)>0)&np.isfinite(b)
            if nod is not None and not (isinstance(nod,float) and np.isnan(nod)):
                vm&=(b!=nod)
            # nodata ignored inside kernel means
            b0=np.where(vm,b,0).astype(np.float32,copy=False)
            vf=vm.astype(np.float32,copy=False)
            out[f"B{bi}"]=b.ravel()[idx]
            for k in ks:
                num=uniform_filter(b0,size=int(k),mode="mirror").astype(np.float32,copy=False)
                den=uniform_filter(vf,size=int(k),mode="mirror").astype(np.float32,copy=False)
                f=np.full((h,w),np.nan,np.float32)
                m=den>0
                f[m]=num[m]/den[m]
                out[f"B{bi}_W{k}"]=f.ravel()[idx]
                del num,den,f
                gc.collect()
            print(f"Filters made for b{bi}")
            del b,vm,b0,vf
            gc.collect()
    return pd.DataFrame(out)[cols]
def run_workflow_csvs_grids(Bathy_Dir,S2_dir,Out_dir_main,one_img=False,n_jobs=-1,make_samples=True,min_depth=0,max_depth=20):
    Bathy_Dir,S2_dir,Out_dir_main=Path(Bathy_Dir),Path(S2_dir),Path(Out_dir_main); Out_dir_main.mkdir(parents=True,exist_ok=True)
    bidx=bathy_index(Bathy_Dir); print(f"Indexed bathy shapefiles: {len(bidx)} | {Bathy_Dir}")
    if isinstance(min_depth,(int,np.integer,float,np.floating)) and isinstance(max_depth,(int,np.integer,float,np.floating)): print(f"Depth range: {min_depth} to {max_depth}")
    s2s=list_tifs(S2_dir)
    if not s2s: raise FileNotFoundError(f"No .tif files found in: {S2_dir}")
    if one_img: s2s=s2s[:1]; print("Mode: one_img=True (processing first .tif only)")
    out=[]; timing=[]
    for s2 in s2s:
        t0=time.perf_counter()
        print(f"\nStarting S2: {s2.name}")
        code=s2_code(s2); od=Out_dir_main/s2.stem; od.mkdir(parents=True,exist_ok=True)
        shp=bidx.get(code)
        if shp is None:
            print(f"  SKIP: no bathy shapefile for code={code}")
            out.append({"s2":s2,"code":code,"out_dir":od,"bathy":None,"bathy_rast":None,"grid_1km":None,"samples":None})
            timing.append({"s2":s2.name,"code":code,"seconds":time.perf_counter()-t0,"status":"skip_no_bathy"}); continue
        print(f"  MATCH: {shp.name}")
        print("  Rasterizing bathy points -> mean per S2 pixel..."); br=bathy_mean_raster(shp,s2,od,code,nodata=-1,n_jobs=n_jobs,min_depth=min_depth,max_depth=max_depth); print(f"  WROTE: {br.name}")
        print("  Making 1km random grid (1..5)..."); g1=rand_1km_grid(br,od,code,cell_m=1000,seed=0); print(f"  WROTE: {g1.name}")
        df=None
        if make_samples:
            print("  Sampling rasters + window avgs -> dataframe...")
            df=sample_rasts_with_avgs(s2,br,g1,kernel_sizes=(3,5,7,9,11,13,15),max_bands=5,bathy_nodata=-1,min_depth=min_depth,max_depth=max_depth)
            df.to_csv(od/f"Data_for_analysis_{code}.csv",index=False); print(f"  WROTE: Data_for_analysis_{code}.csv (rows={len(df)})")
        out.append({"s2":s2,"code":code,"out_dir":od,"bathy":shp,"bathy_rast":br,"grid_1km":g1,"samples":df})
        elapsed=time.perf_counter()-t0
        print(f"  Finished S2: {s2.name} in {elapsed:.2f} seconds")
        timing.append({"s2":s2.name,"code":code,"seconds":elapsed,"status":"ok"})
    if timing:
        pd.DataFrame(timing).to_csv(Out_dir_main/"CsvsGrids_image_timing.csv",index=False)
        print(f"\nWROTE: {Out_dir_main/'CsvsGrids_image_timing.csv'}")
    return out

In [3]:
# === Cell 3: XGBRF + STUMPF training ===
def find_analysis_csv(site_dir:Path):
    c=sorted(site_dir.glob("Data_for_analysis*.csv"))
    return c[0] if c else None
def load_analysis_df(csv_path:Path):
    df=pd.read_csv(csv_path)
    g="group" if "group" in df.columns else ("Group" if "Group" in df.columns else None)
    y="depth" if "depth" in df.columns else ("Depth" if "Depth" in df.columns else None)
    if g is None or y is None: raise ValueError(f"Missing group/depth columns in {csv_path.name}")
    ll=[c for c in ["lon","lat"] if c in df.columns]
    feat=[c for c in df.columns if c not in ([g,y]+ll)]
    df=df[[g,y]+ll+feat].copy().dropna(subset=[g,y]+ll).dropna(subset=feat)
    df[g]=df[g].astype(int); df[y]=df[y].astype(np.float32)
    return df,g,y,feat
# --- XGBRF: test only number of trees (leave-one-group-out MAE) ---
def _max_features_to_colsample(mf, n_feat):
    if mf == "auto": frac = 1.0
    elif mf == "sqrt": frac = np.sqrt(n_feat) / n_feat
    elif mf == "log2": frac = np.log2(n_feat) / n_feat
    else: raise ValueError(f"Unknown max_features: {mf}")
    return float(np.clip(frac, 1.0 / n_feat, 1.0))

def tune_xgbrf_trees_mae(df,group_col,y_col,feat_cols,n_trees_list=(30,100,300,1000),max_features_list=("auto","sqrt","log2"),seed=100):
    X=df[feat_cols].to_numpy(np.float32); y=df[y_col].to_numpy(np.float32); groups=df[group_col].to_numpy()
    lon=df["lon"].to_numpy(np.float64) if "lon" in df.columns else None
    lat=df["lat"].to_numpy(np.float64) if "lat" in df.columns else None
    n_feat=X.shape[1]
    uniq=np.unique(groups); best_mae=np.inf; best_t=None; best_mf=None; best_val=None; rows=[]
    for n_trees in n_trees_list:
        for mf in max_features_list:
            colsample=_max_features_to_colsample(mf,n_feat)
            print(f"Testing XGBRF: n_estimators={n_trees}, max_features={mf} (colsample_bynode={colsample:.4f})")
            pred_rows=[]
            for g in uniq:
                tr=groups!=g; va=groups==g
                if tr.sum()==0 or va.sum()==0: continue
                m=XGBRFRegressor(n_estimators=int(n_trees),colsample_bynode=colsample,n_jobs=-1,random_state=seed)  # only change trees + max_features + n_jobs + seed
                m.fit(X[tr],y[tr]); pred=m.predict(X[va]).astype(np.float32,copy=False)
                lo,la=(lon[va],lat[va]) if lon is not None else (None,None)
                for i,(t,p) in enumerate(zip(y[va],pred)):
                    row={group_col:int(g),"True":float(t),"Predicted":float(p)}
                    if lo is not None: row["lon"],row["lat"]=float(lo[i]),float(la[i])
                    pred_rows.append(row)
            val_df=pd.DataFrame(pred_rows)
            mae=float(mean_absolute_error(val_df["True"],val_df["Predicted"])) if len(val_df) else np.inf
            rows.append({"n_estimators":int(n_trees),"max_features":mf,"colsample_bynode":colsample,"MAE":mae,"n_val":len(val_df)})
            print(f"  Overall MAE: {mae:.4f} (n_val={len(val_df)})")
            if mae<best_mae: best_mae=mae; best_t=int(n_trees); best_mf=mf; best_val=val_df.copy()
    print(f"Best XGBRF: n_estimators={best_t} | max_features={best_mf} | MAE={best_mae:.4f}")
    best_colsample=_max_features_to_colsample(best_mf,n_feat)
    best_model=XGBRFRegressor(n_estimators=int(best_t),colsample_bynode=best_colsample,n_jobs=-1,random_state=seed); best_model.fit(X,y)
    return best_model,best_val,pd.DataFrame(rows),{"best_mae":best_mae,"best_trees":best_t,"best_max_features":best_mf,"n":len(df),"n_groups":len(uniq),"n_feat":n_feat}
# --- STUMPF: grid search n (leave-one-group-out MAE), uses only Blue=B2 and Green=B3 ---
def tune_stumpf_n_mae(df,group_col="group",y_col="depth",n_list=(500,1000,1500),seed=100):
    if "B2" not in df.columns or "B3" not in df.columns: return None,None,None,{"skip":True,"reason":"Missing B2/B3"}
    blue=df["B2"].to_numpy(np.float32); green=df["B3"].to_numpy(np.float32)
    y=df[y_col].to_numpy(np.float32); groups=df[group_col].to_numpy(); uniq=np.unique(groups)
    lon=df["lon"].to_numpy(np.float64) if "lon" in df.columns else None
    lat=df["lat"].to_numpy(np.float64) if "lat" in df.columns else None
    base=np.isfinite(blue)&np.isfinite(green)&np.isfinite(y)&(blue>0)&(green>0)  # log requires positive
    if base.sum()==0: return None,None,None,{"skip":True,"reason":"No valid (B2,B3,depth) >0 rows"}
    blue,green,y,groups=blue[base],green[base],y[base],groups[base]
    if lon is not None: lon,lat=lon[base],lat[base]
    uniq=np.unique(groups)
    best_mae=np.inf; best_n=None; best_val=None; score_rows=[]
    for n in n_list:
        print(f"Testing STUMPF: n={n}")
        x=(np.log(n*blue)/np.log(n*green)).astype(np.float32); pred_rows=[]
        for g in uniq:
            tr=groups!=g; va=groups==g
            if tr.sum()==0 or va.sum()==0: continue
            reg=LinearRegression(); reg.fit(x[tr].reshape(-1,1),y[tr].reshape(-1,1))
            pred=reg.predict(x[va].reshape(-1,1)).ravel().astype(np.float32,copy=False)
            lo,la=(lon[va],lat[va]) if lon is not None else (None,None)
            for i,(t,p) in enumerate(zip(y[va],pred)):
                row={group_col:int(g),"True":float(t),"Predicted":float(p),"n":int(n)}
                if lo is not None: row["lon"],row["lat"]=float(lo[i]),float(la[i])
                pred_rows.append(row)
        val_df=pd.DataFrame(pred_rows)
        mae=float(mean_absolute_error(val_df["True"],val_df["Predicted"])) if len(val_df) else np.inf
        score_rows.append({"n":int(n),"MAE":mae,"n_val":len(val_df)})
        print(f"  Overall MAE: {mae:.4f} (n_val={len(val_df)})")
        if mae<best_mae: best_mae=mae; best_n=int(n); best_val=val_df.copy()
    # fit best equation on ALL valid rows
    xbest=(np.log(best_n*blue)/np.log(best_n*green)).astype(np.float32).reshape(-1,1)
    reg=LinearRegression(); reg.fit(xbest,y.reshape(-1,1))
    slope=float(reg.coef_[0][0]); intercept=float(reg.intercept_[0])
    print(f"Best STUMPF: n={best_n} | MAE={best_mae:.4f} | Depth = {slope:.5f} * log_ratio + {intercept:.5f}")
    meta={"skip":False,"best_mae":best_mae,"best_n":best_n,"slope":slope,"intercept":intercept,"n_rows_valid":int(len(y)),"n_groups":int(len(np.unique(groups)))}
    return (slope,intercept,best_n),best_val,pd.DataFrame(score_rows),meta
def run_workflow_rf_stumpf(Out_dir_main,one_img=False,seed=100,n_trees_list=(30,100,300,1000),max_features_list=("auto","sqrt","log2"),stumpf_n_list=(500,1000,1500)):
    Out_dir_main=Path(Out_dir_main)
    sites=[p for p in sorted(Out_dir_main.iterdir()) if p.is_dir()]
    if one_img: sites=sites[:1]; print("Mode: one_img=True (processing first site dir only)")
    all_summ=[]; timing=[]
    for sd in sites:
        t0=time.perf_counter()
        csvp=find_analysis_csv(sd)
        if csvp is None:
            print(f"\nSKIP {sd.name}: no Data_for_analysis*.csv")
            timing.append({"site":sd.name,"seconds":time.perf_counter()-t0,"status":"skip_no_csv"}); continue
        print(f"\n=== SITE: {sd.name} | csv={csvp.name} ===")
        df,g,y,feat=load_analysis_df(csvp); print(f"Rows={len(df)} | Groups={df[g].nunique()} | Features={len(feat)}")
        # --- XGBRF ---
        model,val_df,tree_df,meta=tune_xgbrf_trees_mae(df,g,y,feat,n_trees_list=n_trees_list,max_features_list=max_features_list,seed=seed)
        joblib.dump(model,sd/"XGBRF_depth_model.joblib")
        val_df.to_csv(sd/"XGBRF_val_predictions.csv",index=False)
        tree_df.to_csv(sd/"XGBRF_tree_scores.csv",index=False)
        print("WROTE: XGBRF_depth_model.joblib | XGBRF_val_predictions.csv | XGBRF_tree_scores.csv")
        # --- STUMPF (run after XGBRF finishes) ---
        stumpf_model,st_val,st_scores,st_meta=tune_stumpf_n_mae(df,group_col=g,y_col=y,n_list=stumpf_n_list,seed=seed)
        if st_meta.get("skip",False):
            print(f"STUMPF SKIP: {st_meta.get('reason','unknown')}")
            st_best_mae=np.nan; st_best_n=np.nan; st_slope=np.nan; st_int=np.nan
        else:
            st_scores.to_csv(sd/"STUMPF_n_scores.csv",index=False)
            st_val.to_csv(sd/"STUMPF_val_predictions.csv",index=False)
            with open(sd/"STUMPF_depth_model.txt","w") as f:
                f.write(f"Best n: {st_meta['best_n']}\n")
                f.write(f"Depth = {st_meta['slope']:.5f} * log_ratio + {st_meta['intercept']:.5f}\n")
                f.write("log_ratio = log(n*Blue)/log(n*Green), Blue=B2, Green=B3\n")
                f.write(f"CV MAE (leave-one-group-out): {st_meta['best_mae']:.6f}\n")
            print("WROTE: STUMPF_n_scores.csv | STUMPF_val_predictions.csv | STUMPF_depth_model.txt")
            st_best_mae=st_meta["best_mae"]; st_best_n=st_meta["best_n"]; st_slope=st_meta["slope"]; st_int=st_meta["intercept"]
        all_summ.append({"site":sd.name,"csv":csvp.name,"rows":meta["n"],"groups":meta["n_groups"],"features":meta["n_feat"],"xgbrf_best_mae":meta["best_mae"],"xgbrf_best_n_estimators":meta["best_trees"],"xgbrf_best_max_features":meta["best_max_features"],"stumpf_best_mae":st_best_mae,"stumpf_best_n":st_best_n,"stumpf_slope":st_slope,"stumpf_intercept":st_int})
        elapsed=time.perf_counter()-t0
        print(f"  Finished grid search: {sd.name} in {elapsed:.2f} seconds")
        timing.append({"site":sd.name,"seconds":elapsed,"status":"ok"})
    all_df=pd.DataFrame(all_summ)
    if len(all_df): all_df.to_csv(Out_dir_main/"Model_compare_all_sites.csv",index=False); print(f"\nWROTE: {Out_dir_main/'Model_compare_all_sites.csv'}")
    if timing:
        pd.DataFrame(timing).to_csv(Out_dir_main/"GridSearch_image_timing.csv",index=False)
        print(f"WROTE: {Out_dir_main/'GridSearch_image_timing.csv'}")
    return all_df


In [4]:
# === Cell 4: prediction rasters ===
def parse_stumpf_txt(p:Path):
    n=s=i=None
    for line in p.read_text().splitlines():
        m=re.search(r"Best n:\s*(\d+)",line); 
        if m: n=int(m.group(1))
        m=re.search(r"Depth\s*=\s*([-\d\.eE]+)\s*\*\s*log_ratio\s*\+\s*([-\d\.eE]+)",line)
        if m: s=float(m.group(1)); i=float(m.group(2))
    if None in (n,s,i): raise ValueError(f"Could not parse STUMPF model: {p}")
    return n,s,i
def find_bathy_rast(d:Path):
    c=sorted(d.glob("Bathy_rast_*.tif"))
    return c[0] if c else None
def _bathy_ok(bathy:Path):
    with rasterio.open(bathy) as b: a=b.read(1).astype(np.float32); nod=b.nodata
    m=np.isfinite(a)
    if nod is not None and not (isinstance(nod,float) and np.isnan(nod)): m&=(a!=nod)
    return m
def build_bad_mask(s2:Path,nb=5,exclude_px=15):
    with rasterio.open(s2) as src:
        h,w=src.height,src.width; nod=src.nodata; bands=list(range(1,min(nb,src.count)+1)); bad=np.zeros((h,w),bool)
        for _,win in src.block_windows(1):
            a=src.read(bands,window=win).astype(np.float32,copy=False); m=np.zeros((win.height,win.width),bool)
            for i,bi in enumerate(bands):
                mm=(src.read_masks(bi,window=win)==0)|(~np.isfinite(a[i]))
                if nod is not None and not (isinstance(nod,float) and np.isnan(nod)): mm|=(a[i]==nod)
                m|=mm
            bad[win.row_off:win.row_off+win.height,win.col_off:win.col_off+win.width]=m
    if exclude_px>0: bad=binary_dilation(bad,structure=np.ones((3,3),bool),iterations=int(exclude_px))
    b=int(exclude_px)
    if b>0: bad[:b,:]=bad[-b:,:]=bad[:,:b]=bad[:,-b:]=True
    return bad
def _pad(win,pad,h,w):
    r0=max(0,int(win.row_off)-pad); c0=max(0,int(win.col_off)-pad)
    r1=min(h,int(win.row_off+win.height)+pad); c1=min(w,int(win.col_off+win.width)+pad)
    return Window(c0,r0,c1-c0,r1-r0),int(win.row_off-r0),int(win.col_off-c0)
def feat_cols_from_csv(csvp:Path):
    cols=list(pd.read_csv(csvp,nrows=0).columns)
    g="group" if "group" in cols else ("Group" if "Group" in cols else None)
    y="depth" if "depth" in cols else ("Depth" if "Depth" in cols else None)
    if g is None or y is None: raise ValueError(f"Missing group/depth columns in {csvp.name}")
    return [c for c in cols if c not in {g,y}]
def find_analysis_csv(site_dir:Path):
    c=sorted(site_dir.glob("Data_for_analysis*.csv"))
    return c[0] if c else None
def predict_xgbrf_raster(s2:Path,bathy:Path,model,feat_cols,out_tif:Path,kernel_sizes=(3,5,7,9,11,13,15),max_bands=5,out_nodata=-9999.0):
    ks=list(kernel_sizes); pad=int(max(ks)//2)
    bathy_ok=_bathy_ok(bathy)
    with rasterio.open(s2) as src:
        prof=src.profile.copy(); prof.update(driver="GTiff",count=1,dtype="float32",nodata=float(out_nodata),compress="ZSTD",predictor=2,BIGTIFF="IF_SAFER")
        bands=list(range(1,min(max_bands,src.count)+1)); nod=src.nodata
        with rasterio.open(out_tif,"w",**prof) as dst:
            for _,win in src.block_windows(1):
                out=np.full((win.height,win.width),out_nodata,np.float32)
                wpad,ri,ci=_pad(win,pad,src.height,src.width); rr=slice(ri,ri+win.height); cc=slice(ci,ci+win.width)
                a=src.read(bands,window=wpad).astype(np.float32,copy=False); v=[]; feat={}
                good=bathy_ok[win.row_off:win.row_off+win.height,win.col_off:win.col_off+win.width].copy()
                for i,bi in enumerate(bands):
                    vm=(src.read_masks(bi,window=wpad)>0)&np.isfinite(a[i])
                    if nod is not None and not (isinstance(nod,float) and np.isnan(nod)): vm&=(a[i]!=nod)
                    good&=vm[rr,cc]
                    v.append(vm.astype(np.float32,copy=False))
                    a[i]=np.where(vm,a[i],0).astype(np.float32,copy=False)
                    feat[f"B{bi}"]=a[i,rr,cc]
                if not good.any(): dst.write(out,1,window=win); continue
                for i,bi in enumerate(bands):
                    for k in ks:
                        s=uniform_filter(a[i],size=int(k),mode="mirror").astype(np.float32,copy=False)
                        c=uniform_filter(v[i],size=int(k),mode="mirror").astype(np.float32,copy=False)
                        f=np.full_like(s,np.nan,np.float32); m=c>0; f[m]=s[m]/c[m]
                        feat[f"B{bi}_W{k}"]=f[rr,cc]; del s,c,f
                    gc.collect()
                good1=good.ravel().copy(); vec=[]
                for c in feat_cols:
                    vv=feat[c].ravel(); good1&=np.isfinite(vv); vec.append(vv)
                if good1.any():
                    X=np.column_stack([vv[good1] for vv in vec]).astype(np.float32,copy=False)
                    out.ravel()[good1]=model.predict(X).astype(np.float32,copy=False)
                dst.write(out,1,window=win)
    for bi in bands: print(f"Filters made for b{bi}")
    return out_tif
def predict_stumpf_raster(s2:Path,bathy:Path,st_txt:Path,out_tif:Path,kernel_sizes=(3,5,7,9,11,13,15),out_nodata=-9999.0):
    n,slope,inter=parse_stumpf_txt(st_txt)
    bathy_ok=_bathy_ok(bathy)
    with rasterio.open(s2) as src:
        prof=src.profile.copy(); prof.update(driver="GTiff",count=1,dtype="float32",nodata=float(out_nodata),compress="ZSTD",predictor=2,BIGTIFF="IF_SAFER")
        nod=src.nodata
        with rasterio.open(out_tif,"w",**prof) as dst:
            for _,win in src.block_windows(1):
                out=np.full((win.height,win.width),out_nodata,np.float32)
                b2=src.read(2,window=win).astype(np.float32,copy=False)
                b3=src.read(3,window=win).astype(np.float32,copy=False)
                ok=bathy_ok[win.row_off:win.row_off+win.height,win.col_off:win.col_off+win.width]
                ok&=(src.read_masks(2,window=win)>0)&(src.read_masks(3,window=win)>0)&np.isfinite(b2)&np.isfinite(b3)
                if nod is not None and not (isinstance(nod,float) and np.isnan(nod)): ok&=(b2!=nod)&(b3!=nod)
                ok&=(b2>0)&(b3>0)
                lr=np.full((win.height,win.width),np.nan,np.float32); lr[ok]=np.log(n*b2[ok])/np.log(n*b3[ok])
                m=np.isfinite(lr); out[m]=(slope*lr[m]+inter).astype(np.float32,copy=False)
                dst.write(out,1,window=win)
    return out_tif
def run_predict_workflow_predictions(S2_dir,Out_dir_main,one_img=False,kernel_sizes=(3,5,7,9,11,13,15),max_bands=5):
    S2_dir,Out_dir_main=Path(S2_dir),Path(Out_dir_main)
    s2s=sorted(S2_dir.glob("S2*.tif"))
    if not s2s: raise FileNotFoundError(f"No S2*.tif found in: {S2_dir}")
    if one_img: s2s=s2s[:1]; print("Mode: one_img=True (processing first S2 tif only)")
    timing=[]
    for s2 in s2s:
        t0=time.perf_counter()
        od=Out_dir_main/s2.stem
        if not od.is_dir():
            print(f"\nSKIP {s2.name}: missing {od}")
            timing.append({"s2":s2.name,"seconds":time.perf_counter()-t0,"status":"skip_no_dir"}); continue
        csvp=find_analysis_csv(od); bathy=find_bathy_rast(od)
        if csvp is None or bathy is None:
            print(f"\nSKIP {s2.name}: missing Data_for_analysis*.csv or Bathy_rast_*.tif in {od.name}")
            timing.append({"s2":s2.name,"seconds":time.perf_counter()-t0,"status":"skip_missing_files"}); continue
        feat_cols=feat_cols_from_csv(csvp)
        xgbp=od/"XGBRF_depth_model.joblib"; stp=od/"STUMPF_depth_model.txt"
        if not xgbp.exists() or not stp.exists():
            print(f"\nSKIP {s2.name}: missing model files in {od.name}")
            timing.append({"s2":s2.name,"seconds":time.perf_counter()-t0,"status":"skip_missing_models"}); continue
        model=joblib.load(xgbp)
        feat_cols=[c for c in feat_cols if c not in {"lon","lat"}]
        print(f"\n=== PREDICT: {s2.name} -> {od.name} ===")
        predict_xgbrf_raster(s2,bathy,model,feat_cols,od/f"Pred_XGBRF_{s2.stem}.tif",kernel_sizes=kernel_sizes,max_bands=max_bands)
        predict_stumpf_raster(s2,bathy,stp,od/f"Pred_STUMPF_{s2.stem}.tif",kernel_sizes=kernel_sizes)
        elapsed=time.perf_counter()-t0
        print(f"Done in {elapsed:.2f} seconds.")
        timing.append({"s2":s2.name,"seconds":elapsed,"status":"ok"})
    if timing:
        pd.DataFrame(timing).to_csv(Out_dir_main/"Predictions_image_timing.csv",index=False)
        print(f"\nWROTE: {Out_dir_main/'Predictions_image_timing.csv'}")
def merge_all_timing(Out_dir_main):
    Out_dir_main=Path(Out_dir_main)
    def _load(name,seconds_col):
        p=Out_dir_main/name
        if not p.exists(): return pd.DataFrame(columns=["site",seconds_col])
        d=pd.read_csv(p)
        key=d["s2"] if "s2" in d.columns else d["site"]
        key=key.astype(str).str.replace(r"\.tif$","",regex=True)
        return pd.DataFrame({"site":key,seconds_col:d["seconds"]})
    a=_load("CsvsGrids_image_timing.csv","image_timing_seconds")
    b=_load("GridSearch_image_timing.csv","grid_search_seconds")
    c=_load("Predictions_image_timing.csv","prediction_seconds")
    merged=a.merge(b,on="site",how="outer").merge(c,on="site",how="outer")
    for col in ("image_timing_seconds","grid_search_seconds","prediction_seconds"):
        if col not in merged.columns: merged[col]=np.nan
    merged["total_seconds"]=merged[["image_timing_seconds","grid_search_seconds","prediction_seconds"]].sum(axis=1,skipna=True)
    merged=merged.sort_values("site").reset_index(drop=True)
    out=Out_dir_main/"All_stages_image_timing.csv"
    merged.to_csv(out,index=False)
    print(f"WROTE: {out}")
    return merged


In [5]:
# === Cell 4: run ===
Out_dir_main=r"E:\GIS\SDB_Compare\Composite_Results\UBtimer"
Bathy_Dir=r"E:\GIS\SDB_Compare\Bathy_shps"
S2_dir=r"E:\GIS\SDB_Compare\COMPOSITES\UBtimer"
Out_dir_main=Path(Out_dir_main); log=[]; start_dt=datetime.now(); t_all=time.perf_counter()
log.append(f"Start time: {start_dt:%Y-%m-%d %H:%M:%S}")
for name, fn, kwargs in [
    ("run_workflow_csvs_grids", run_workflow_csvs_grids, dict(Bathy_Dir=Bathy_Dir, S2_dir=S2_dir, Out_dir_main=Out_dir_main, one_img=False, n_jobs=-1, make_samples=True, min_depth=0, max_depth=20)),
    ("run_workflow_rf_stumpf", run_workflow_rf_stumpf, dict(Out_dir_main=Out_dir_main, one_img=False, seed=100, n_trees_list=(30,100,300,1000), max_features_list=("auto","sqrt","log2"), stumpf_n_list=(500,1000,1500))),
    ("run_predict_workflow_predictions", run_predict_workflow_predictions, dict(S2_dir=S2_dir, Out_dir_main=Out_dir_main, one_img=False, kernel_sizes=(3,5,7,9,11,13,15), max_bands=5)),
]:
    t=time.perf_counter(); fn(**kwargs); log.append(f"{name}: {time.perf_counter()-t:.2f} seconds")
merge_all_timing(Out_dir_main)
end_dt=datetime.now(); total=time.perf_counter()-t_all
log += [f"End time: {end_dt:%Y-%m-%d %H:%M:%S}", f"Total duration: {int(total//3600):02d}:{int((total%3600)//60):02d}:{total%60:05.2f}"]
log_path=Out_dir_main/"workflow_timing_log.txt"
log_path.write_text("\n".join(log), encoding="utf-8")
print("\n".join(log), f"\n\nTiming log written to: {log_path}")

Indexed bathy shapefiles: 4 | E:\GIS\SDB_Compare\Bathy_shps
Depth range: 0 to 20

Starting S2: S2_UB_BestImage_S2B_MSIL2A_20200710T160829_N0500_R140_T19VDG_20230603T070409_plusSCL_masked.tif
  MATCH: UB_bathy_prj.shp
  Rasterizing bathy points -> mean per S2 pixel...
  WROTE: Bathy_rast_UB.tif
  Making 1km random grid (1..5)...
  WROTE: 1kmGrid_UB.tif
  Sampling rasters + window avgs -> dataframe...
Samples kept: 94615 (depth_ok=94626, center_bad=121687, kept=94615)
Filters made for b1
Filters made for b2
Filters made for b3
Filters made for b4
Filters made for b5
  WROTE: Data_for_analysis_UB.csv (rows=94615)
  Finished S2: S2_UB_BestImage_S2B_MSIL2A_20200710T160829_N0500_R140_T19VDG_20230603T070409_plusSCL_masked.tif in 13.13 seconds

Starting S2: S2_UB_cleancoastal_median_composite.tif
  MATCH: UB_bathy_prj.shp
  Rasterizing bathy points -> mean per S2 pixel...
  WROTE: Bathy_rast_UB.tif
  Making 1km random grid (1..5)...
  WROTE: 1kmGrid_UB.tif
  Sampling rasters + window avgs -> d

In [6]:
# === Cell 4: run ===
Out_dir_main=r"E:\GIS\SDB_Compare\Composite_Results\KWtimer"
Bathy_Dir=r"E:\GIS\SDB_Compare\Bathy_shps"
S2_dir=r"E:\GIS\SDB_Compare\COMPOSITES\KWtimer"
Out_dir_main=Path(Out_dir_main); log=[]; start_dt=datetime.now(); t_all=time.perf_counter()
log.append(f"Start time: {start_dt:%Y-%m-%d %H:%M:%S}")
for name, fn, kwargs in [
    ("run_workflow_csvs_grids", run_workflow_csvs_grids, dict(Bathy_Dir=Bathy_Dir, S2_dir=S2_dir, Out_dir_main=Out_dir_main, one_img=False, n_jobs=-1, make_samples=True, min_depth=0, max_depth=20)),
    ("run_workflow_rf_stumpf", run_workflow_rf_stumpf, dict(Out_dir_main=Out_dir_main, one_img=False, seed=100, n_trees_list=(30,100,300,1000), max_features_list=("auto","sqrt","log2"), stumpf_n_list=(500,1000,1500))),
    ("run_predict_workflow_predictions", run_predict_workflow_predictions, dict(S2_dir=S2_dir, Out_dir_main=Out_dir_main, one_img=False, kernel_sizes=(3,5,7,9,11,13,15), max_bands=5)),
]:
    t=time.perf_counter(); fn(**kwargs); log.append(f"{name}: {time.perf_counter()-t:.2f} seconds")
merge_all_timing(Out_dir_main)
end_dt=datetime.now(); total=time.perf_counter()-t_all
log += [f"End time: {end_dt:%Y-%m-%d %H:%M:%S}", f"Total duration: {int(total//3600):02d}:{int((total%3600)//60):02d}:{total%60:05.2f}"]
log_path=Out_dir_main/"workflow_timing_log.txt"
log_path.write_text("\n".join(log), encoding="utf-8")
print("\n".join(log), f"\n\nTiming log written to: {log_path}")

Indexed bathy shapefiles: 4 | E:\GIS\SDB_Compare\Bathy_shps
Depth range: 0 to 20

Starting S2: S2_KW_BestImage_S2B_MSIL2A_20191130T160619_N0500_R097_T17RMH_20230704T080201_plusSCL_masked.tif
  MATCH: KW_bathy_prj.shp
  Rasterizing bathy points -> mean per S2 pixel...
  WROTE: Bathy_rast_KW.tif
  Making 1km random grid (1..5)...
  WROTE: 1kmGrid_KW.tif
  Sampling rasters + window avgs -> dataframe...
Samples kept: 1044770 (depth_ok=1044854, center_bad=118, kept=1044770)
Filters made for b1
Filters made for b2
Filters made for b3
Filters made for b4
Filters made for b5
  WROTE: Data_for_analysis_KW.csv (rows=1044770)
  Finished S2: S2_KW_BestImage_S2B_MSIL2A_20191130T160619_N0500_R097_T17RMH_20230704T080201_plusSCL_masked.tif in 163.71 seconds

Starting S2: S2_KW_cleancoastal_median_composite.tif
  MATCH: KW_bathy_prj.shp
  Rasterizing bathy points -> mean per S2 pixel...
  WROTE: Bathy_rast_KW.tif
  Making 1km random grid (1..5)...
  WROTE: 1kmGrid_KW.tif
  Sampling rasters + window avg

In [7]:
# === Cell 4: run ===
Out_dir_main = r"E:\GIS\SDB_Compare\Composite_Results\Cortimer"
Bathy_Dir = r"E:\GIS\SDB_Compare\Bathy_shps"
S2_dir = r"E:\GIS\SDB_Compare\COMPOSITES\Cortimer"
Out_dir_main = Path(Out_dir_main);
log = [];
start_dt = datetime.now();
t_all = time.perf_counter()
log.append(f"Start time: {start_dt:%Y-%m-%d %H:%M:%S}")
for name, fn, kwargs in [
    ("run_workflow_csvs_grids", run_workflow_csvs_grids,
     dict(Bathy_Dir=Bathy_Dir, S2_dir=S2_dir, Out_dir_main=Out_dir_main, one_img=False, n_jobs=-1, make_samples=True,
          min_depth=0, max_depth=20)),
    ("run_workflow_rf_stumpf", run_workflow_rf_stumpf, dict(Out_dir_main=Out_dir_main, one_img=False, seed=100, n_trees_list=(30,100,300,1000), max_features_list=("auto","sqrt","log2"), stumpf_n_list=(500,1000,1500))),
    ("run_predict_workflow_predictions", run_predict_workflow_predictions,
     dict(S2_dir=S2_dir, Out_dir_main=Out_dir_main, one_img=False, kernel_sizes=(3, 5, 7, 9, 11, 13, 15), max_bands=5)),
]:
    t = time.perf_counter();
    fn(**kwargs);
    log.append(f"{name}: {time.perf_counter() - t:.2f} seconds")
merge_all_timing(Out_dir_main)
end_dt = datetime.now();
total = time.perf_counter() - t_all
log += [f"End time: {end_dt:%Y-%m-%d %H:%M:%S}",
        f"Total duration: {int(total // 3600):02d}:{int((total % 3600) // 60):02d}:{total % 60:05.2f}"]
log_path = Out_dir_main / "workflow_timing_log.txt"
log_path.write_text("\n".join(log), encoding="utf-8")
print("\n".join(log), f"\n\nTiming log written to: {log_path}")

Indexed bathy shapefiles: 4 | E:\GIS\SDB_Compare\Bathy_shps
Depth range: 0 to 20

Starting S2: S2_COR_BestImage_S2B_MSIL2A_20170409T101019_N0500_R022_T32TNL_20231110T192027_plusSCL_masked.tif
  MATCH: COR_bathy_prj.shp
  Rasterizing bathy points -> mean per S2 pixel...
  WROTE: Bathy_rast_COR.tif
  Making 1km random grid (1..5)...
  WROTE: 1kmGrid_COR.tif
  Sampling rasters + window avgs -> dataframe...
Samples kept: 427968 (depth_ok=428502, center_bad=1037, kept=427968)
Filters made for b1
Filters made for b2
Filters made for b3
Filters made for b4
Filters made for b5
  WROTE: Data_for_analysis_COR.csv (rows=427968)
  Finished S2: S2_COR_BestImage_S2B_MSIL2A_20170409T101019_N0500_R022_T32TNL_20231110T192027_plusSCL_masked.tif in 29.67 seconds

Starting S2: S2_COR_cleancoastal_median_composite.tif
  MATCH: COR_bathy_prj.shp
  Rasterizing bathy points -> mean per S2 pixel...
  WROTE: Bathy_rast_COR.tif
  Making 1km random grid (1..5)...
  WROTE: 1kmGrid_COR.tif
  Sampling rasters + wind

In [8]:
# === Cell 4: run ===
Out_dir_main=r"E:\GIS\SDB_Compare\Composite_Results\CCtimer"
Bathy_Dir=r"E:\GIS\SDB_Compare\Bathy_shps"
S2_dir=r"E:\GIS\SDB_Compare\COMPOSITES\CCtimer"
Out_dir_main=Path(Out_dir_main); log=[]; start_dt=datetime.now(); t_all=time.perf_counter()
log.append(f"Start time: {start_dt:%Y-%m-%d %H:%M:%S}")
for name, fn, kwargs in [
    ("run_workflow_csvs_grids", run_workflow_csvs_grids, dict(Bathy_Dir=Bathy_Dir, S2_dir=S2_dir, Out_dir_main=Out_dir_main, one_img=False, n_jobs=-1, make_samples=True, min_depth=0, max_depth=20)),
    ("run_workflow_rf_stumpf", run_workflow_rf_stumpf, dict(Out_dir_main=Out_dir_main, one_img=False, seed=100, n_trees_list=(30,100,300,1000), max_features_list=("auto","sqrt","log2"), stumpf_n_list=(500,1000,1500))),
    ("run_predict_workflow_predictions", run_predict_workflow_predictions, dict(S2_dir=S2_dir, Out_dir_main=Out_dir_main, one_img=False, kernel_sizes=(3,5,7,9,11,13,15), max_bands=5)),
]:
    t=time.perf_counter(); fn(**kwargs); log.append(f"{name}: {time.perf_counter()-t:.2f} seconds")
merge_all_timing(Out_dir_main)
end_dt=datetime.now(); total=time.perf_counter()-t_all
log += [f"End time: {end_dt:%Y-%m-%d %H:%M:%S}", f"Total duration: {int(total//3600):02d}:{int((total%3600)//60):02d}:{total%60:05.2f}"]
log_path=Out_dir_main/"workflow_timing_log.txt"
log_path.write_text("\n".join(log), encoding="utf-8")
print("\n".join(log), f"\n\nTiming log written to: {log_path}")

Indexed bathy shapefiles: 4 | E:\GIS\SDB_Compare\Bathy_shps
Depth range: 0 to 20

Starting S2: S2_CC_BestImage_S2B_MSIL2A_20190425T153609_N0500_R111_T19TCG_20221023T115432_plusSCL_masked.tif
  MATCH: CC_bathy_prj.shp
  Rasterizing bathy points -> mean per S2 pixel...
  WROTE: Bathy_rast_CC.tif
  Making 1km random grid (1..5)...
  WROTE: 1kmGrid_CC.tif
  Sampling rasters + window avgs -> dataframe...
Samples kept: 497863 (depth_ok=499050, center_bad=1484, kept=497863)
Filters made for b1
Filters made for b2
Filters made for b3
Filters made for b4
Filters made for b5
  WROTE: Data_for_analysis_CC.csv (rows=497863)
  Finished S2: S2_CC_BestImage_S2B_MSIL2A_20190425T153609_N0500_R111_T19TCG_20221023T115432_plusSCL_masked.tif in 77.74 seconds

Starting S2: S2_CC_cleancoastal_median_composite.tif
  MATCH: CC_bathy_prj.shp
  Rasterizing bathy points -> mean per S2 pixel...
  WROTE: Bathy_rast_CC.tif
  Making 1km random grid (1..5)...
  WROTE: 1kmGrid_CC.tif
  Sampling rasters + window avgs ->